In [ ]:
import numpy
import os
import pandas

import datetime as _dt
import pyproj as _proj

### Process AADF

In [ ]:
traffic_flow_df = pandas.read_csv('../input/ukTrafficAADF.csv')

In [ ]:
traffic_flow_df.head()

In [ ]:
src_prj = _proj.Proj("+init=EPSG:4326")
dst_prj = _proj.Proj("+init=EPSG:3857")

x_coords = traffic_flow_df.Lon.values
y_coords = traffic_flow_df.Lat.values
x_coords, y_coords = _proj.transform(src_prj, dst_prj, x_coords, y_coords)
traffic_flow_df["x"] = x_coords
traffic_flow_df["y"] = y_coords
traffic_flow_df["z"] = 0

Create a new "stacked" dataframe, where each count point, volume measurement, and year, is a separate record.

In [ ]:
volume_columns = \
[
    "CarsTaxis",
    "BusesCoaches",
    "Motorcycles",
    "PedalCycles",
    "LightGoodsVehicles",
    "V2AxleRigidHGV",
    "V3AxleRigidHGV",
    "V4or5AxleRigidHGV",
    "V3or4AxleArticHGV",
    "V5AxleArticHGV",
    "V6orMoreAxleArticHGV"
]

dfs = []
for i, name in enumerate(volume_columns):
    df = traffic_flow_df[["CP", "AADFYear", name]].copy()
    df.columns = ["CP", "AADFYear", "Volume"]
    df["Traffic_Mode"] = i + 1
    dfs.append(df)

stacked_volumes_df = pandas.concat(dfs)
stacked_volumes_df = stacked_volumes_df.reset_index()

In [ ]:
len(stacked_volumes_df)

In [ ]:
stacked_volumes_df.head()

## Note: The cells below will not run on the Kaggle kernel, as they require the [CityPhi](http://http://www.inrosoftware.com/cityphi) library, which is not part of the kernel image.

See [this page](https://imgur.com/a/7pb9T) for example visuals produced with this notebook.

### Load the data into CityPhi features

In [ ]:
import cityphi.application as _app
import cityphi.attribute as _att
import cityphi.layer as _layer
import cityphi.feature as _feat
import cityphi.widget as _widget

Here, we create a location feature, and create the traffic flow feature by indexing into it. This enables automatic stacking of different volume measurements at the same count location.

In [ ]:
count_location_feat = _feat.PointFeature(
    traffic_flow_df.CP.values,
    traffic_flow_df[["x", "y", "z"]].values)

traffic_flow_feat = _feat.PointFeature.from_points(
    stacked_volumes_df.index.values,
    stacked_volumes_df.CP.values,
    count_location_feat)

In [ ]:
len(count_location_feat)

In [ ]:
len(traffic_flow_feat)

In [ ]:
for name in ["AADFYear", "CP", "Volume", "Traffic_Mode"]:
    traffic_flow_feat.add_attribute(
        name, "int32", stacked_volumes_df.index.values, stacked_volumes_df[name].values)

### Launch the app and set up the visualization

In [ ]:
%gui cityphi
app = _app.Application()
app.graphics_settings.ambient_occlusion = "SSAO_LOW"

In [ ]:
traffic_flow_layer = _layer.PointLayer(traffic_flow_feat)

traffic_flow_layer.name = "Average Annual Daily Flow"
traffic_flow_layer.start_time = _att.FeatureAttribute("AADFYear")
traffic_flow_layer.end_time = _att.FeatureAttribute("AADFYear")
traffic_flow_layer.sort_order = _att.FeatureAttribute("Traffic_Mode")
traffic_flow_layer.stacked = True
traffic_flow_layer.height = _att.MultiplierAttribute(
    0.2, _att.FeatureAttribute("Volume"))
traffic_flow_layer.radius = 400
traffic_flow_layer.min_pixel_size = 4

app.add_layer(traffic_flow_layer)

In [ ]:
app.set_view(app.full_view)

In [ ]:
colors = [
    (166, 206, 227),
    (31, 120, 180),
    (178, 223, 138),
    (51, 160, 44),
    (251, 154, 153),
    (227, 26, 28),
    (253, 191, 111),
    (255, 127, 0),
    (202, 178, 214),
    (106, 61, 154),
    (255, 255, 153)
]
traffic_flow_layer.color = _att.DiscreteColorAttribute(
    colors, _att.FeatureAttribute("Traffic_Mode"), volume_columns,
    values=[[i] for i in range(1, 12)])

In [ ]:
def display_time(t):
    return unicode(int(t))

def change_time(t):
    traffic_flow_layer.time_window = int(t), int(t)
    
time_slider = _widget.TimeSlider(2000, 2015, display_time, change_time)
time_slider.speed = 2
app.add_widget(time_slider)

In [ ]:
traffic_flow_legend = _widget.HTMLWidget(
    "Average Annual Daily Flow<br><br>" + traffic_flow_layer.color._repr_html_())
app.add_widget(traffic_flow_legend)